# 🤖 Entrenamiento de Modelos Baseline — `IsToxic`

## 🎯 Objetivo
Entrenar y evaluar dos modelos baseline de clasificación binaria sobre `IsToxic`,
con tracking completo de experimentos en MLflow/DagsHub.

## 📋 Flujo
1. Conexión con DagsHub + MLflow
2. Carga de datos preprocesados
3. Vectorización TF-IDF
4. Funciones auxiliares de evaluación
5. Logistic Regression
6. LightGBM
7. Comparativa final

> Los CSVs de entrada vienen de `data/processed/V_04/` generados en el notebook
> de preprocesamiento. La vectorización TF-IDF se hace aquí para poder loguear
> sus parámetros en MLflow.

In [5]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import dagshub

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay
)
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

TARGET       = 'IsToxic'
RANDOM_STATE = 42
AVG_METHOD   = 'binary'

os.makedirs('../../reports/figures', exist_ok=True)
os.makedirs('../../models', exist_ok=True)

print('Setup completado.')

Setup completado.


## 🔗 1. Conexión con DagsHub y MLflow

> ⚠️ Ejecutar esta celda cada vez que se reinicie el kernel o se quiera
> loguear un nuevo experimento. El servidor de DagsHub no mantiene la
> conexión indefinidamente.

In [4]:
import os
import dagshub
import mlflow

print('=' * 60)
print('  CONECTANDO CON DAGSHUB - MLFLOW')
print('=' * 60)

# Token compartido del equipo — permite subir experimentos al repo de Gemita
token = "f16c39c4380542f071674b19fac461d44195b694"
os.environ["DATALOOP_TOKEN"]            = token
os.environ["MLFLOW_TRACKING_USERNAME"]  = "gemita284"
os.environ["MLFLOW_TRACKING_PASSWORD"]  = token

dagshub.init(
    repo_owner='gemita284',
    repo_name='Project_9_NLP_Team2',
    mlflow=True
)

mlflow.set_experiment('youtube_toxic_comments')

print('\n✅ Conexión establecida con éxito. ¡Ya podéis registrar modelos!')
print('   Experimento: youtube_toxic_comments')
print('\n   Ver en DagsHub:')
print('   https://dagshub.com/gemita284/Project_9_NLP_Team2.mlflow')

  CONECTANDO CON DAGSHUB - MLFLOW


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f2f1c402-d565-4214-b2d7-ac0ce3dbcf37&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c8de2153ea1cdabb2ac531bf3aacdd82907d3d986d4a195c1d6d622cfa2b6cc2




Accessing as RaulCtm

Initialized MLflow to track repo "gemita284/Project_9_NLP_Team2"

Repository gemita284/Project_9_NLP_Team2 initialized!


✅ Conexión establecida con éxito. ¡Ya podéis registrar modelos!
   Experimento: youtube_toxic_comments

   Ver en DagsHub:
   https://dagshub.com/gemita284/Project_9_NLP_Team2.mlflow


## 📂 2. Carga de datos preprocesados

Se cargan los tres conjuntos generados en el notebook de preprocesamiento.
`test.csv` se carga aquí pero **no se usa hasta la evaluación final**.

In [6]:
# ── CELDA 2 ──────────────────────────────────────────────────────────────────
# commit: "feat: load preprocessed train/val/test splits from data/processed/V_04 (#15, #11)"
train_df = pd.read_csv('../../data/processed/V_04/train_augmented.csv')
val_df   = pd.read_csv('../../data/processed/V_04/val.csv')
test_df  = pd.read_csv('../../data/processed/V_04/test.csv')

print('Conjuntos cargados:')
print(f'  Train : {len(train_df):>4} filas  |  tóxico: {train_df[TARGET].mean():.1%}')
print(f'  Val   : {len(val_df):>4} filas  |  tóxico: {val_df[TARGET].mean():.1%}')
print(f'  Test  : {len(test_df):>4} filas  |  tóxico: {test_df[TARGET].mean():.1%}')

X_train_raw = train_df['text_procesado']
X_val_raw   = val_df['text_procesado']
X_test_raw  = test_df['text_procesado']

y_train = train_df[TARGET].astype(int)
y_val   = val_df[TARGET].astype(int)
y_test  = test_df[TARGET].astype(int)

Conjuntos cargados:
  Train :  741 filas  |  tóxico: 46.6%
  Val   :  150 filas  |  tóxico: 46.0%
  Test  :  150 filas  |  tóxico: 46.0%


## 🔢 3. Vectorización TF-IDF

El vectorizador se ajusta **solo sobre train** y se aplica con `transform`
sobre val y test, simulando el comportamiento real en producción.

Los parámetros se loguearán en cada run de MLflow junto con los del modelo.

| Parámetro | Valor |
|---|---|
| `max_features` | 5 000 |
| `ngram_range` | (1, 2) |
| `min_df` | 2 |
| `max_df` | 0.95 |
| `sublinear_tf` | True |

In [8]:
# ── CELDA 3 ──────────────────────────────────────────────────────────────────
# commit: "feat: fit TF-IDF on train and transform val/test, ready for MLflow logging (#15, #11)"
TFIDF_PARAMS = {
    'max_features': 5000,
    'ngram_range':  (1, 2),
    'min_df':       2,
    'max_df':       0.95,
    'sublinear_tf': True
}

vectorizer = TfidfVectorizer(**TFIDF_PARAMS)

X_train = vectorizer.fit_transform(X_train_raw)
X_val   = vectorizer.transform(X_val_raw)
X_test  = vectorizer.transform(X_test_raw)

print('Dimensiones TF-IDF:')
print(f'  Train: {X_train.shape}')
print(f'  Val  : {X_val.shape}')
print(f'  Test : {X_test.shape}')

# Guardar vectorizador
import joblib
joblib.dump(vectorizer, '../../models/tfidf_vectorizer_IsToxic_V04.pkl')
print('\nVectorizador guardado en models/')

Dimensiones TF-IDF:
  Train: (741, 1998)
  Val  : (150, 1998)
  Test : (150, 1998)

Vectorizador guardado en models/
